# Task 2: Model Comparison, Final Judgement, and Prediction

This notebook collects the exported results. It does not retrain any model.

## 1. Setup

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import confusion_matrix
from torchvision.models import densenet121, efficientnet_b0

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "pyproject.toml").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the repository root")
sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import TEST_IMAGE_DIR, IMAGE_TARGET_SIZE, load_image_array, load_manifest
from src.task2_utils import (
    TASK2_OUTPUT_DIR, TASK2_PREDICTION_PATH, TASK2_SPLIT_PATH,
    evaluate_predictions, extract_visual_features, load_validation_predictions,
)
sns.set_theme(style="whitegrid", context="notebook")


## 2. Compare All Models

In [ ]:
setup = json.loads((TASK2_OUTPUT_DIR / "setup" / "config.json").read_text())
CLASSES = setup["classes"]
TARGET = setup["target"]
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}

frames = []
for folder, label in [
    ("setup", "Baseline: majority class"),
    ("random_forest", "Random Forest"),
    ("efficientnet_b0", "EfficientNet-B0"),
    ("densenet121", "DenseNet-121"),
]:
    path = TASK2_OUTPUT_DIR / folder / "validation_predictions.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Run that model notebook first.")
    frame = load_validation_predictions(path, CLASSES)
    frames.append((label, folder, frame))

rows = []
for label, folder, frame in frames:
    rows.append(evaluate_predictions(
        frame["true_index"].to_numpy(), frame["predicted_index"].to_numpy(),
        frame[[f"score_{name}" for name in CLASSES]].to_numpy(), label,
    ))
model_comparison = pd.DataFrame(rows).sort_values("Macro-F1", ascending=False)
analysis_dir = TASK2_OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
model_comparison.to_csv(analysis_dir / "model_comparison.csv", index=False)
display(model_comparison.style.format({c: "{:.4f}" for c in model_comparison.columns if c != "Model"}))

FINAL_NAME = model_comparison.iloc[0]["Model"]
winner = next(item for item in frames if item[0] == FINAL_NAME)
final_validation = winner[2]
print("Selected model:", FINAL_NAME)


## 3. Error Analysis

In [ ]:
y_true = final_validation["true_index"].to_numpy()
y_pred = final_validation["predicted_index"].to_numpy()
matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)), normalize="true")
plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, vmin=0, vmax=1)
plt.title(f"{FINAL_NAME}: row-normalised confusion matrix")
plt.xlabel("Predicted season")
plt.ylabel("True season")
plt.tight_layout()
plt.show()

errors = final_validation.loc[final_validation["true_index"] != final_validation["predicted_index"]]
print(f"Validation errors: {len(errors):,} / {len(final_validation):,}")
display(errors.head(20))


## 4. Ultimate Judgement

Use the comparison and error evidence above to justify the selected model. Macro-F1 is the primary metric because the season classes are imbalanced.

## 5. Produce the Final Task 2 Prediction

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
template_path = REPO_ROOT / "datasets" / "test" / "styles_prediction.csv"
template = pd.read_csv(template_path)
test_paths = [Path(TEST_IMAGE_DIR) / f"{image_id}.jpg" for image_id in template["id"]]
missing = [path for path in test_paths if not path.exists()]
assert not missing, f"{len(missing)} test images are missing"

def build_neural(name):
    if name == "EfficientNet-B0":
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
        path = REPO_ROOT / "models" / "task2_efficientnet_b0.pt"
    else:
        model = densenet121(weights=None)
        model.classifier = nn.Linear(model.classifier.in_features, len(CLASSES))
        path = REPO_ROOT / "models" / "task2_densenet121.pt"
    blob = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(blob["state_dict"])
    return model.to(DEVICE).eval()

mean = torch.tensor(setup["normalisation_mean"], device=DEVICE).view(1, 3, 1, 1)
std = torch.tensor(setup["normalisation_std"], device=DEVICE).view(1, 3, 1, 1)

if FINAL_NAME == "Random Forest":
    model = joblib.load(REPO_ROOT / "models" / "task2_random_forest.joblib")
    features = np.vstack([extract_visual_features(load_image_array(path, IMAGE_TARGET_SIZE, False))
                          for path in test_paths])
    scores = model.predict_proba(features)
    predicted = model.classes_[scores.argmax(axis=1)]
else:
    model = build_neural(FINAL_NAME)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(test_paths), 256):
            arrays = np.stack([load_image_array(path, IMAGE_TARGET_SIZE, False)
                               for path in test_paths[start:start + 256]])
            images = torch.from_numpy(arrays).to(DEVICE).permute(0, 3, 1, 2).float() / 255
            chunks.append(model((images - mean) / std).float().cpu())
    scores = torch.cat(chunks).numpy()
    predicted = scores.argmax(axis=1)

predictions = template.copy()
predictions[TARGET] = [CLASSES[index] for index in predicted]
TASK2_PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(TASK2_PREDICTION_PATH, index=False)
print("Written:", TASK2_PREDICTION_PATH)
display(predictions.head())
